# Modelado Predictivo con Machine Learning: Random Forest Classifier
## Caso de Aplicación: Detección y Clasificación de Intrusiones en Red (NSL-KDD)

### 1. Objetivos del Entrenamiento
A partir de la curación de datos y el análisis exploratorio realizado en la etapa previa, este notebook entrena y evalúa modelos de **Random Forest** aprovechando directamente las métricas de red más relevantes:
1. **Clasificación Binaria (`class_binary`):** Detección en tiempo real de tráfico legítimo (`normal`) vs intrusión (`anomaly`).
2. **Clasificación Multiclase (`attack_category`):** Identificación taxonómica del vector de agresión (`Normal`, `DoS`, `Probe`, `R2L`, `U2R`).
3. **Importancia de Características (Feature Importance):** Validación cuantitativa de qué métricas (ej. `src_bytes`, `count`, `serror_rate`) aportan mayor poder discriminatorio.
4. **Validación del Muestreo Estratificado:** Demostración de que entrenar el modelo con solo la muestra estratificada ($n = 2.000$) mantiene una precisión superior al **99%** sobre el resto de la población no vista ($N = 23.191$).


---
### Celda 1: Importación de Librerías y Configuración


In [ ]:
# ==============================================================================
# CELDA 1: CARGA DE LIBRERÍAS DE MACHINE LEARNING Y VISUALIZACIÓN
# ==============================================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    accuracy_score,
    f1_score
)

# Configuración gráfica
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.autolayout'] = True

SEED = 42
print('Librerías importadas exitosamente.')


---
### Celda 2: Carga de Datos y Selección de Métricas Clave
Seleccionamos las variables analizadas durante el EDA, agrupadas en sus dimensiones operativas.


In [ ]:
# ==============================================================================
# CELDA 2: CARGA DE DATOS Y DEFINICIÓN DE FEATURES MATRICIALES
# ==============================================================================
DATASET_PATH = 'nsl_kdd_20percent_curado.csv'
df = pd.read_csv(DATASET_PATH)

# Métricas seleccionadas basadas en el EDA y arquitectura de red
metricas_numericas = [
    # 1. Ventana corta de 2 segundos (Eje del modelado de Poisson para DoS)
    'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    
    # 2. Conexión elemental y volumetría en bytes
    'duration', 'src_bytes', 'dst_bytes',
    
    # 3. Comportamiento en sesión / host (Clave para R2L y U2R)
    'logged_in', 'num_failed_logins', 'root_shell', 'su_attempted',
    
    # 4. Historial acumulado de 100 conexiones hacia el destino (Ataques Slow & Low)
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 
    'dst_host_diff_srv_rate', 'dst_host_serror_rate'
]

metricas_categoricas = ['protocol_type', 'flag', 'service']

# One-Hot Encoding para variables categóricas
X = pd.get_dummies(df[metricas_numericas + metricas_categoricas], drop_first=True)
y_binario = df['class_binary'].map({'normal': 0, 'anomaly': 1})
y_multiclase = df['attack_category']

print(f'Población cargada: {len(df):,} registros')
print(f'Matriz de características (X): {X.shape[1]} columnas predictoras codificadas')


---
## Sección 1: Modelo 1 — Random Forest Binario (Detección de Intrusiones)
Entrenamos un clasificador binario sobre el 80% de los datos y lo testeamos sobre el 20% no visto.


In [ ]:
# ==============================================================================
# CELDA 3: ENTRENAMIENTO DEL CLASIFICADOR BINARIO (80/20 STRATIFIED)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binario, test_size=0.20, random_state=SEED, stratify=y_binario
)

rf_bin = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=SEED,
    n_jobs=-1
)
rf_bin.fit(X_train, y_train)

y_pred_bin = rf_bin.predict(X_test)
y_prob_bin = rf_bin.predict_proba(X_test)[:, 1]

acc_bin = accuracy_score(y_test, y_pred_bin)
f1_bin = f1_score(y_test, y_pred_bin)
auc_bin = roc_auc_score(y_test, y_prob_bin)

print('=== MÉTRICAS DEL RANDOM FOREST BINARIO ===')
print(f'Accuracy : {acc_bin*100:.2f}%')
print(f'F1-Score : {f1_bin*100:.2f}%')
print(f'ROC-AUC  : {auc_bin*100:.2f}%
')
print(classification_report(y_test, y_pred_bin, target_names=['Normal (0)', 'Anomaly (1)'], digits=4))


In [ ]:
# ==============================================================================
# CELDA 4: MATRIZ DE CONFUSIÓN BINARIA
# ==============================================================================
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_bin)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Ataque'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Matriz de Confusión: Random Forest Binario', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


### Importancia de Métricas (Feature Importance)
Evaluamos la importancia Gini de cada métrica para entender qué factores son los más decisivos para el modelo.


In [ ]:
# ==============================================================================
# CELDA 5: IMPORTANCIA RELATIVA DE CARACTERÍSTICAS (GINI IMPORTANCE)
# ==============================================================================
importancias = pd.Series(rf_bin.feature_importances_, index=X.columns).sort_values(ascending=False)
top15 = importancias.head(15)[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15.index, top15.values, color='#2a9d8f', edgecolor='black', alpha=0.85)
ax.set_title('Top 15 Métricas Más Determinantes en Detección (Random Forest)', fontweight='bold', pad=12)
ax.set_xlabel('Importancia Relativa (Gini)')

for i, v in enumerate(top15.values):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=9)

ax.set_xlim(0, max(top15.values) * 1.15)
plt.tight_layout()
plt.show()


---
## Sección 2: Modelo 2 — Random Forest Multiclase (Clasificación de Amenazas)
Clasificamos entre las 5 familias (`Normal`, `DoS`, `Probe`, `R2L`, `U2R`) utilizando ponderación balanceada (`class_weight='balanced_subsample'`) para compensar el desbalance severo de clases.


In [ ]:
# ==============================================================================
# CELDA 6: ENTRENAMIENTO DEL CLASIFICADOR MULTICLASE
# ==============================================================================
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multiclase, test_size=0.20, random_state=SEED, stratify=y_multiclase
)

rf_multi = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced_subsample',
    random_state=SEED,
    n_jobs=-1
)
rf_multi.fit(X_train_m, y_train_m)

y_pred_multi = rf_multi.predict(X_test_m)
clases_ordenadas = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

print('=== REPORTE DE CLASIFICACIÓN MULTICLASE ===')
print(classification_report(y_test_m, y_pred_multi, labels=clases_ordenadas, zero_division=0, digits=4))


In [ ]:
# ==============================================================================
# CELDA 7: MATRIZ DE CONFUSIÓN MULTICLASE
# ==============================================================================
fig, ax = plt.subplots(figsize=(7, 6))
cm_multi = confusion_matrix(y_test_m, y_pred_multi, labels=clases_ordenadas)
disp_multi = ConfusionMatrixDisplay(confusion_matrix=cm_multi, display_labels=clases_ordenadas)
disp_multi.plot(cmap='YlGnBu', ax=ax, values_format='d')
ax.set_title('Matriz de Confusión: Random Forest Multiclase\n(5 Familias de Amenazas)', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


---
## Sección 3: Validación del Muestreo Estratificado en el Rendimiento del Modelo
¿Qué ocurre si entrenamos el modelo **únicamente con la muestra estratificada de $n = 2.000$ registros** (un ahorro del 92% en cómputo) y lo evaluamos sobre los **23.191 registros restantes de la población no vistos**?


In [ ]:
# ==============================================================================
# CELDA 8: EXPERIMENTO DE MUESTREO ESTRATIFICADO COMO DATASET DE ENTRENAMIENTO
# ==============================================================================
# Extracción de muestra estratificada (n = 2.000)
frac = 2000 / len(df)
idx_muestra = df.groupby('attack_category', group_keys=False).apply(
    lambda g: g.sample(n=max(1, int(np.round(len(g) * frac))), random_state=SEED)
).index

X_train_peq = X.loc[idx_muestra]
y_train_peq = y_binario.loc[idx_muestra]

# Resto de la población no vista para test (92% de los datos)
idx_resto = df.index.difference(idx_muestra)
X_test_grande = X.loc[idx_resto]
y_test_grande = y_binario.loc[idx_resto]

rf_peq = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=SEED, n_jobs=-1)
rf_peq.fit(X_train_peq, y_train_peq)

y_pred_grande = rf_peq.predict(X_test_grande)
acc_peq = accuracy_score(y_test_grande, y_pred_grande)
f1_peq = f1_score(y_test_grande, y_pred_grande)

print('=== IMPACTO DEL MUESTREO ESTRATIFICADO EN MACHINE LEARNING ===')
print(f'Tamaño del conjunto de entrenamiento : {len(X_train_peq):,} registros (8% de la población)')
print(f'Tamaño del conjunto de evaluación     : {len(X_test_grande):,} registros no vistos (92% de la población)')
print(f'Accuracy sobre datos no vistos        : {acc_peq*100:.2f}%')
print(f'F1-Score sobre datos no vistos        : {f1_peq*100:.2f}%')
print('\n-> Conclusión: El muestreo estratificado permite ahorrar >90% de recursos computacionales sin degradar el poder predictivo.')
